# Depth Calibration Visualization
Compare alpha-beta parameters across relative/metric modes

In [30]:
import numpy as np

path = "/nfs/stak/users/sanchej7/hpc-share/Computer_Vision/Data/trunk_spurs/depth_fused/bark_brown/lpy_envy_00000/lpy_envy_00000_shot03_fused.npy"

arr = np.load(path)

print("Shape:", arr.shape)

ImportError: 
Importing the multiarray numpy extension module failed.  Most
likely you are trying to import a failed build of numpy.
If you're working with a numpy git repo, try `git clean -xdf` (removes all
files not under version control).  Otherwise reinstall numpy.

Original error was: cannot import name multiarray


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_ROOT = Path('/nfs/stak/users/sanchej7/hpc-share/Computer_Vision/Data/full_trunk')

# Load calibration parameters
with open(DATA_ROOT / 'alpha_beta.json') as f:
    rel = json.load(f)
with open(DATA_ROOT / 'alpha_beta_metric.json') as f:
    met = json.load(f)

# Load evaluation results
with open(DATA_ROOT / 'evaluation_results/evaluation_results_relative.json') as f:
    rel_eval = json.load(f)
with open(DATA_ROOT / 'evaluation_results/evaluation_results_metric.json') as f:
    met_eval = json.load(f)

## Global Parameters

In [ ]:
print(f"Relative: α={rel['global']['alpha']:.4f}, β={rel['global']['beta']:.4f}")
print(f"Metric:   α={met['global']['alpha']:.4f}, β={met['global']['beta']:.4f}")

## Per-Shot Calibration

In [ ]:
shots = sorted(rel['per_shot'].keys())
x = np.arange(len(shots))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Alpha
axes[0].bar(x - w/2, [rel['per_shot'][s]['alpha'] for s in shots], w, 
            label='Relative', color='steelblue', alpha=0.8)
axes[0].bar(x + w/2, [met['per_shot'][s]['alpha'] for s in shots], w, 
            label='Metric', color='coral', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(shots, rotation=45)
axes[0].set_ylabel('Alpha (slope)')
axes[0].set_title('Per-Shot Alpha')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Beta
axes[1].bar(x - w/2, [rel['per_shot'][s]['beta'] for s in shots], w, 
            label='Relative', color='steelblue', alpha=0.8)
axes[1].bar(x + w/2, [met['per_shot'][s]['beta'] for s in shots], w, 
            label='Metric', color='coral', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(shots, rotation=45)
axes[1].set_ylabel('Beta (offset)')
axes[1].set_title('Per-Shot Beta')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Metric Performance Comparison

In [ ]:
# Compare global calibration performance
metrics = ['abs_rel', 'rmse', 'silog', 'r2', 'a1']
labels = ['AbsRel↓', 'RMSE↓', 'SILog↓', 'R²↑', 'δ<1.25↑']

rel_vals = [rel_eval['granularity_comparison']['Global'][m] for m in metrics]
met_vals = [met_eval['granularity_comparison']['Global'][m] for m in metrics]

x = np.arange(len(metrics))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, rel_vals, w, label='Relative', color='steelblue', alpha=0.8)
ax.bar(x + w/2, met_vals, w, label='Metric', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Metric Value')
ax.set_title('Global Calibration Performance')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRelative vs Metric (Global Calibration):")
for label, m, r_val, m_val in zip(labels, metrics, rel_vals, met_vals):
    print(f"  {label:12s}: Rel={r_val:.4f}, Met={m_val:.4f}")

## Pro_images Comparison (if available)

In [ ]:
# Check if Pro_images results exist
if rel_eval.get('pro_images_comparison') and met_eval.get('pro_images_comparison'):
    calib_types = ['Uncalibrated', 'Global', 'Per-shot', 'Per-tree']
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    x = np.arange(len(calib_types))
    w = 0.35
    
    # R² comparison
    rel_r2 = [rel_eval['pro_images_comparison'][c]['r2'] for c in calib_types]
    met_r2 = [met_eval['pro_images_comparison'][c]['r2'] for c in calib_types]
    
    axes[0].bar(x - w/2, rel_r2, w, label='Relative', color='steelblue', alpha=0.8)
    axes[0].bar(x + w/2, met_r2, w, label='Metric', color='coral', alpha=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(calib_types, rotation=45, ha='right')
    axes[0].set_ylabel('R² (higher is better)')
    axes[0].set_title('Pro_images Correlation (R²)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # AbsRel comparison
    rel_absrel = [rel_eval['pro_images_comparison'][c]['abs_rel'] for c in calib_types]
    met_absrel = [met_eval['pro_images_comparison'][c]['abs_rel'] for c in calib_types]
    
    axes[1].bar(x - w/2, rel_absrel, w, label='Relative', color='steelblue', alpha=0.8)
    axes[1].bar(x + w/2, met_absrel, w, label='Metric', color='coral', alpha=0.8)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(calib_types, rotation=45, ha='right')
    axes[1].set_ylabel('AbsRel (lower is better)')
    axes[1].set_title('Pro_images Absolute Relative Error')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Pro_images comparison results not available.")
    print("Run evaluation with --compare-pro flag to generate.")